In [1]:
import h5py
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import norm
from tqdm.auto import tqdm

/home/lenka-hake/Documents/ALS/ALS-Assignment-3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Constants

In [2]:
number_of_anchors = 137
prediction_horizon = 16
number_of_bins = 1000

minimum_probability = 1e-12
default_standard_deviation = 0.03

anchor_index_array = np.arange(number_of_anchors)
training_size_array = np.ceil(16 * (2 ** (anchor_index_array / 8.0))).astype(float)

bin_edge_array = np.linspace(0.0, 1.0, number_of_bins + 1)
bin_center_array = 0.5 * (bin_edge_array[:-1] + bin_edge_array[1:])

## Data Loading

In [3]:
def load_lcdb_file(file_path):
    with h5py.File(file_path, "r") as file_handle:
        error_rate_array = np.asarray(file_handle["error rate"][:], dtype=float)
        learner_array = np.asarray(file_handle["learner"][:]).reshape(-1)

    if learner_array.dtype.kind in {"S", "U", "O"}:
        cleaned_learner_array = []
        for learner_value in learner_array:
            if isinstance(learner_value, bytes):
                cleaned_learner_array.append(learner_value.decode())
            else:
                cleaned_learner_array.append(str(learner_value))
        learner_array = np.asarray(cleaned_learner_array)
    else:
        learner_array = learner_array.astype(int)

    return error_rate_array, learner_array

## Helpers

In [4]:
def clip_to_unit_interval(value):
    return np.clip(value, 0.0, 1.0)

def get_contiguous_observed_prefix(curve_array):
    observed_anchor_list = []
    observed_value_list = []

    for anchor_index, error_value in enumerate(curve_array):
        if np.isnan(error_value):
            break
        observed_anchor_list.append(anchor_index)
        observed_value_list.append(float(error_value))

    if len(observed_anchor_list) == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    observed_anchor_array = np.asarray(observed_anchor_list, dtype=int)
    observed_value_array = np.asarray(observed_value_list, dtype=float)
    return observed_anchor_array, observed_value_array


def get_valid_training_examples(
    error_rate_array,
    learner_array,
    prediction_horizon=16,
    minimum_prefix_length=6,
    maximum_examples_per_curve=3,
    random_seed=42,
):
    random_number_generator = np.random.default_rng(random_seed)
    example_list = []

    for curve_index in range(len(error_rate_array)):
        curve_array = error_rate_array[curve_index]
        learner_value = learner_array[curve_index]

        observed_anchor_array, observed_value_array = get_contiguous_observed_prefix(curve_array)
        observed_length = len(observed_anchor_array)

        if observed_length < minimum_prefix_length + prediction_horizon:
            continue

        possible_prefix_end_array = np.arange(minimum_prefix_length - 1, observed_length - prediction_horizon)

        if len(possible_prefix_end_array) == 0:
            continue

        if len(possible_prefix_end_array) > maximum_examples_per_curve:
            chosen_prefix_end_array = np.sort(
                random_number_generator.choice(
                    possible_prefix_end_array,
                    size=maximum_examples_per_curve,
                    replace=False,
                )
            )
        else:
            chosen_prefix_end_array = possible_prefix_end_array

        for prefix_end_index in chosen_prefix_end_array:
            target_anchor_index = prefix_end_index + prediction_horizon

            example_list.append(
                {
                    "curve_index": curve_index,
                    "learner_value": learner_value,
                    "observed_anchor_array": observed_anchor_array[: prefix_end_index + 1].copy(),
                    "observed_value_array": observed_value_array[: prefix_end_index + 1].copy(),
                    "target_anchor_index": int(observed_anchor_array[target_anchor_index]),
                    "target_value": float(observed_value_array[target_anchor_index]),
                    "prefix_end_anchor_index": int(observed_anchor_array[prefix_end_index]),
                }
            )

    return example_list

## Parametric Models

In [5]:
def power_law_model(training_size, parameter_a, parameter_b, parameter_c):
    return parameter_a * np.power(training_size, -parameter_b) + parameter_c

def logarithmic_model(training_size, parameter_a, parameter_b):
    return parameter_a + parameter_b * np.log(training_size)

def inverse_model(training_size, parameter_a, parameter_b, parameter_c):
    return parameter_c + parameter_a / (training_size + parameter_b)

In [6]:
parametric_model_dictionary = {
    "power_law": {
        "function": power_law_model,
        "initial_parameter_function": lambda x_values, y_values: [
            max(y_values[0] - y_values[-1], 0.001),
            0.3,
            float(y_values[-1])
        ],
        "minimum_points": 4
    },
    "logarithmic": {
        "function": logarithmic_model,
        "initial_parameter_function": lambda x_values, y_values: [
            float(y_values[0]),
            -0.05
        ],
        "minimum_points": 3
    },
    "inverse": {
        "function": inverse_model,
        "initial_parameter_function": lambda x_values, y_values: [
            max(y_values[0] - y_values[-1], 0.001),
            1.0,
            float(y_values[-1])
        ],
        "minimum_points": 4
    }
}

## Fitting and Prediction

In [7]:
def fit_and_predict_parametric_model(
    observed_anchor_index_array,
    observed_error_array,
    target_anchor_index,
    model_name,
    use_anchor_sizes=True
):

    model_specification = parametric_model_dictionary[model_name]
    model_function = model_specification["function"]
    minimum_points = model_specification["minimum_points"]

    if len(observed_error_array) < minimum_points:
        return float(np.clip(observed_error_array[-1], 0.0, 1.0))

    if use_anchor_sizes:
        x_values = training_size_array[observed_anchor_index_array]
        target_value = training_size_array[target_anchor_index]
    else:
        x_values = observed_anchor_index_array.astype(float)
        target_value = float(target_anchor_index)

    y_values = observed_error_array.astype(float)

    mask = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = x_values[mask]
    y_values = y_values[mask]

    if len(y_values) < minimum_points:
        return float(np.clip(observed_error_array[-1], 0.0, 1.0))

    try:
        initial_parameters = model_specification["initial_parameter_function"](x_values, y_values)

        fit_result = curve_fit(
            model_function,
            x_values,
            y_values,
            p0=initial_parameters,
            maxfev=20000
        )

        fitted_parameters = fit_result[0]

        predicted_value = model_function(
            np.array([target_value]),
            *fitted_parameters
        )[0]

        predicted_value = float(np.clip(predicted_value, 0.0, 1.0))

        if not np.isfinite(predicted_value):
            predicted_value = float(np.clip(observed_error_array[-1], 0.0, 1.0))

        return predicted_value

    except Exception:
        return float(np.clip(observed_error_array[-1], 0.0, 1.0))

## Validation Example Generation

In [8]:
def build_validation_examples(
    curve_array,
    learner_array,
    horizon=16,
    minimum_prefix_points=6,
    maximum_examples_per_curve=3,
    random_seed=42
):

    random_generator = np.random.default_rng(random_seed)
    example_list = []

    for curve_index in range(len(curve_array)):

        curve = curve_array[curve_index]
        learner = learner_array[curve_index]

        anchor_index_array, error_array = get_contiguous_observed_prefix(curve)

        prefix_length = len(anchor_index_array)

        if prefix_length < minimum_prefix_points + horizon:
            continue

        possible_prefix_endpoints = np.arange(
            minimum_prefix_points - 1,
            prefix_length - horizon
        )

        if len(possible_prefix_endpoints) > maximum_examples_per_curve:
            selected_endpoints = random_generator.choice(
                possible_prefix_endpoints,
                size=maximum_examples_per_curve,
                replace=False
            )
        else:
            selected_endpoints = possible_prefix_endpoints

        for prefix_endpoint in selected_endpoints:

            observed_anchor_index_array = anchor_index_array[:prefix_endpoint + 1]
            observed_error_array = error_array[:prefix_endpoint + 1]

            target_anchor_index = anchor_index_array[prefix_endpoint + horizon]
            target_error_value = error_array[prefix_endpoint + horizon]

            example_list.append({
                "learner": learner,
                "observed_anchor_index_array": observed_anchor_index_array,
                "observed_error_array": observed_error_array,
                "target_anchor_index": target_anchor_index,
                "target_error_value": target_error_value
            })

    return example_list

## Model Evaluation

In [9]:
def evaluate_parametric_models(example_list, model_name_list):

    result_row_list = []

    for example in tqdm(example_list):

        learner = example["learner"]
        target_error_value = example["target_error_value"]

        for model_name in model_name_list:

            prediction = fit_and_predict_parametric_model(
                example["observed_anchor_index_array"],
                example["observed_error_array"],
                example["target_anchor_index"],
                model_name
            )

            absolute_error = abs(prediction - target_error_value)

            result_row_list.append({
                "learner": learner,
                "model_name": model_name,
                "absolute_error": absolute_error,
                "success": 1
            })

    result_dataframe = pd.DataFrame(result_row_list)

    return result_dataframe

## Determining the Best Model for Each Learner

In [10]:
def select_best_model_per_learner(result_dataframe):

    grouped_dataframe = (
        result_dataframe
        .groupby(["learner", "model_name"])
        .agg(mean_absolute_error=("absolute_error", "mean"))
        .reset_index()
    )

    best_model_rows = (
        grouped_dataframe
        .sort_values(["learner", "mean_absolute_error"])
        .groupby("learner")
        .first()
        .reset_index()
    )

    best_model_dictionary = dict(
        zip(best_model_rows["learner"], best_model_rows["model_name"])
    )

    return best_model_dictionary, best_model_rows

## Evaluation of Model Residuals

In [11]:
def compute_residual_standard_deviation(
    example_list,
    best_model_dictionary
):

    residual_value_dictionary = {}

    for example in tqdm(example_list):

        learner = example["learner"]
        model_name = best_model_dictionary[learner]

        prediction = fit_and_predict_parametric_model(
            example["observed_anchor_index_array"],
            example["observed_error_array"],
            example["target_anchor_index"],
            model_name
        )

        residual = prediction - example["target_error_value"]

        key = (learner, model_name)

        if key not in residual_value_dictionary:
            residual_value_dictionary[key] = []

        residual_value_dictionary[key].append(residual)

    residual_dictionary = {}

    for key, residual_list in residual_value_dictionary.items():
        residual_array = np.array(residual_list, dtype=float)
        standard_deviation = float(np.std(residual_array))

        if not np.isfinite(standard_deviation) or standard_deviation < 0.01:
            standard_deviation = 0.01

        residual_dictionary[key] = standard_deviation

    return residual_dictionary

## Conversion of Gaussian Distribution to Bin Probabilities

In [12]:
def gaussian_to_bin_probabilities(mean_value, standard_deviation):

    if standard_deviation is None or not np.isfinite(standard_deviation) or standard_deviation < 0.01:
        standard_deviation = 0.03

    bin_edge_array = np.linspace(0.0, 1.0, 1001)

    probability_array = norm.cdf(
        bin_edge_array[1:],
        loc=mean_value,
        scale=standard_deviation
    ) - norm.cdf(
        bin_edge_array[:-1],
        loc=mean_value,
        scale=standard_deviation
    )

    probability_array = np.clip(probability_array, 0.0, None)

    probability_sum = probability_array.sum()

    if probability_sum <= 0.0 or not np.isfinite(probability_sum):
        probability_array = np.ones(1000, dtype=float) / 1000.0
    else:
        probability_array = probability_array / probability_sum

    return probability_array

## Evaluation of Test Set Probabilities

In [13]:
def predict_test_probabilities(
    test_curve_array,
    test_learner_array,
    best_model_dictionary,
    residual_dictionary
):

    probability_row_list = []

    for curve_index in tqdm(range(len(test_curve_array))):

        curve = test_curve_array[curve_index]
        learner = test_learner_array[curve_index]

        observed_anchor_index_array, observed_error_array = get_contiguous_observed_prefix(curve)

        if len(observed_anchor_index_array) == 0:
            probability_row_list.append(np.ones(1000, dtype=float) / 1000.0)
            continue

        target_anchor_index = observed_anchor_index_array[-1] + 16

        if target_anchor_index >= len(training_size_array):
            target_anchor_index = len(training_size_array) - 1

        model_name = best_model_dictionary.get(learner, "power_law")

        prediction = fit_and_predict_parametric_model(
            observed_anchor_index_array,
            observed_error_array,
            target_anchor_index,
            model_name
        )

        standard_deviation = residual_dictionary.get((learner, model_name), 0.03)

        probability_array = gaussian_to_bin_probabilities(
            prediction,
            standard_deviation
        )

        probability_row_list.append(probability_array)

    probability_matrix = np.vstack(probability_row_list)

    return probability_matrix

## Submission Generation

In [14]:
def write_submission_file(probability_matrix, output_path):

    submission_dataframe = pd.DataFrame(
        probability_matrix,
        columns=[f"bin {bin_index}" for bin_index in range(1000)]
    )

    submission_dataframe.insert(0, "id", np.arange(len(submission_dataframe)))

    submission_dataframe.to_csv(output_path, index=False)

    print("Saved submission to:", output_path)

## Full Pipeline

In [16]:
train_curve_array, train_learner_array = load_lcdb_file("data/LCDB11_ER_train.hdf5")
test_curve_array, test_learner_array = load_lcdb_file("data/LCDB11_ER_eval.hdf5")

example_list = build_validation_examples(
    train_curve_array,
    train_learner_array,
    horizon=16,
    minimum_prefix_points=6,
    maximum_examples_per_curve=3,
    random_seed=42
)

print("Number of validation examples:", len(example_list))

model_name_list = list(parametric_model_dictionary.keys())

result_dataframe = evaluate_parametric_models(
    example_list,
    model_name_list
)

print(result_dataframe.head())

best_model_dictionary, best_model_rows = select_best_model_per_learner(
    result_dataframe
)

print(best_model_rows)

residual_dictionary = compute_residual_standard_deviation(
    example_list,
    best_model_dictionary
)

probability_matrix = predict_test_probabilities(
    test_curve_array,
    test_learner_array,
    best_model_dictionary,
    residual_dictionary
)

print("Probability matrix shape:", probability_matrix.shape)
print("First row sum:", probability_matrix[0].sum())

write_submission_file(
    probability_matrix,
    "parametric_submission.csv"
)

Number of validation examples: 120384


  0%|          | 0/120384 [00:00<?, ?it/s]/tmp/ipykernel_25576/552261791.py:2: RuntimeWarning: overflow encountered in power
  return parameter_a * np.power(training_size, -parameter_b) + parameter_c
  0%|          | 65/120384 [00:01<40:03, 50.05it/s]/tmp/ipykernel_25576/2329504655.py:35: OptimizeWarning: Covariance of the parameters could not be estimated
  fit_result = curve_fit(
  8%|▊         | 9417/120384 [02:52<33:58, 54.45it/s]  


KeyboardInterrupt: 